In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

project_root = Path(r"D:\Projects\AgriRisk and ROI Prediction\Dump")

raw_file = project_root / "data" / "raw" / "upag" / "upag_complete_2023_25.csv"

upag_harmonized_file = (
    project_root / "data" / "processed" / "upag_harmonized.csv"
)

unified_file = (
    project_root / "data" / "processed" / "unified"
    / "unified_crop_yield_2013_2025.csv"
)

output_dir = project_root / "data" / "processed" / "upag"
output_dir.mkdir(parents=True, exist_ok=True)

print("Project root:", project_root)
print("Raw file:", raw_file)
print("Existing UPAg file:", upag_harmonized_file)
print("Existing unified file:", unified_file)

Project root: D:\Projects\AgriRisk and ROI Prediction\Dump
Raw file: D:\Projects\AgriRisk and ROI Prediction\Dump\data\raw\upag\upag_complete_2023_25.csv
Existing UPAg file: D:\Projects\AgriRisk and ROI Prediction\Dump\data\processed\upag_harmonized.csv
Existing unified file: D:\Projects\AgriRisk and ROI Prediction\Dump\data\processed\unified\unified_crop_yield_2013_2025.csv


In [2]:
#Loading complete upag dataset
df = pd.read_csv(raw_file)

print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

Shape: (38277, 10)

Columns:
['State', 'District', 'Crop', 'Season', 'Area-2023-24', 'Area-2024-25', 'Production-2023-24', 'Production-2024-25', 'Yield-2023-24', 'Yield-2024-25']

First 5 rows:


,State,District,Crop,Season,Area-2023-24,Area-2024-25,Production-2023-24,Production-2024-25,Yield-2023-24,Yield-2024-25
0,Andaman And Nicobar Islands,Nicobars,Rice,Kharif,0.2,3.5,0.45,2.57,2266.0,735.0
1,Andaman And Nicobar Islands,Nicobars,Rice,Total,0.2,3.5,0.45,2.57,2266.0,735.0
2,Andaman And Nicobar Islands,Nicobars,Cereals,Kharif,0.2,3.5,0.45,2.57,2266.0,735.0
3,Andaman And Nicobar Islands,Nicobars,Cereals,Total,0.2,3.5,0.45,2.57,2266.0,735.0
4,Andaman And Nicobar Islands,Nicobars,Total Food Grains,Kharif,0.2,3.5,0.45,2.57,2266.0,735.0


In [4]:
#Inspecting crops and seasons
print("Number of unique crops:", df["Crop"].nunique())

print("\nCrops:")
print(sorted(df["Crop"].dropna().unique()))

print("\nSeasons:")
print(df["Season"].value_counts())

Number of unique crops: 37

Crops:
['Bajra', 'Barley', 'Castorseed', 'Cereals', 'Cotton', 'Gram', 'Groundnut', 'Guarseed', 'Jowar', 'Jute', 'Jute & Mesta', 'Lentil', 'Linseed', 'Maize', 'Mesta', 'Moong', 'Nigerseed', 'Nutri/Coarse Cereals', 'Other Pulses', 'Ragi', 'Rapeseed & Mustard', 'Rice', 'Safflower', 'Sannhemp', 'Sesamum', 'Shree Anna /Nutri Cereals', 'Small Millets', 'Soybean', 'Sugarcane', 'Sunflower', 'Tobacco', 'Total Food Grains', 'Total Oil Seeds', 'Total Pulses', 'Tur', 'Urad', 'Wheat']

Seasons:
Season
Total     16433
Kharif    11521
Rabi       7279
Summer     3044
Name: count, dtype: int64


In [5]:
#Selecting the 8 missing crops
target_crops = [
    "Tur",
    "Bajra",
    "Gram",
    "Groundnut",
    "Jowar",
    "Ragi",
    "Soybean",
    "Sugarcane"
]

recent_8 = df[
    df["Crop"].isin(target_crops) &
    df["Season"].eq("Total")
].copy()

print("Selected rows:", len(recent_8))

print("\nCrop counts:")
print(recent_8["Crop"].value_counts().sort_index())

print("\nYears available:")
print("2023-24 and 2024-25")

Selected rows: 3686

Crop counts:
Crop
Bajra        430
Gram         535
Groundnut    510
Jowar        432
Ragi         326
Soybean      352
Sugarcane    551
Tur          550
Name: count, dtype: int64

Years available:
2023-24 and 2024-25


In [6]:
#Checking that each crop has both the years
print(
    recent_8[
        ["Crop", "Area-2023-24", "Area-2024-25",
         "Production-2023-24", "Production-2024-25",
         "Yield-2023-24", "Yield-2024-25"]
    ].isna().sum()
)

print("\nRows per crop:")
print(
    recent_8.groupby("Crop").size().sort_index()
)

Crop                    0
Area-2023-24          320
Area-2024-25          473
Production-2023-24    337
Production-2024-25    482
Yield-2023-24         337
Yield-2024-25         482
dtype: int64

Rows per crop:
Crop
Bajra        430
Gram         535
Groundnut    510
Jowar        432
Ragi         326
Soybean      352
Sugarcane    551
Tur          550
dtype: int64


In [7]:
#Converting wide data to long format
records = []

for _, row in recent_8.iterrows():

    for year in ["2023-24", "2024-25"]:

        records.append({
            "year": year,
            "state": row["State"],
            "district": row["District"],
            "crop": row["Crop"],
            "season": "Annual",
            "area_ha": row[f"Area-{year}"],
            "production_tonnes": row[f"Production-{year}"],
            "yield_kg_ha": row[f"Yield-{year}"],
            "source": "UPAg"
        })

recent_long = pd.DataFrame(records)

print("Shape:", recent_long.shape)

print("\nColumns:")
print(recent_long.columns.tolist())

print("\nYears:")
print(recent_long["year"].value_counts().sort_index())

Shape: (7372, 9)

Columns:
['year', 'state', 'district', 'crop', 'season', 'area_ha', 'production_tonnes', 'yield_kg_ha', 'source']

Years:
year
2023-24    3686
2024-25    3686
Name: count, dtype: int64


In [8]:
#Standardizing crop names
crop_mapping = {
    "Tur": "Arhar/Tur",
    "Bajra": "Bajra",
    "Gram": "Gram",
    "Groundnut": "Groundnut",
    "Jowar": "Jowar",
    "Ragi": "Ragi",
    "Soybean": "Soyabean",
    "Sugarcane": "Sugarcane"
}

recent_long["crop"] = recent_long["crop"].replace(crop_mapping)

print("Standardized crops:")
print(sorted(recent_long["crop"].unique()))

Standardized crops:
['Arhar/Tur', 'Bajra', 'Gram', 'Groundnut', 'Jowar', 'Ragi', 'Soyabean', 'Sugarcane']


In [9]:
#Validating units using yeild
check = recent_long[
    recent_long["area_ha"].notna() &
    recent_long["production_tonnes"].notna() &
    recent_long["yield_kg_ha"].notna() &
    (recent_long["area_ha"] > 0)
].copy()

check["calculated_yield"] = (
    check["production_tonnes"] * 1000 / check["area_ha"]
)

check["yield_difference"] = (
    check["calculated_yield"] - check["yield_kg_ha"]
).abs()

print("Valid yield checks:", len(check))

print("\nMean absolute yield difference:",
      check["yield_difference"].mean())

print("Maximum absolute yield difference:",
      check["yield_difference"].max())

print("\nSample validation:")
display(
    check[
        ["crop", "area_ha", "production_tonnes",
         "yield_kg_ha", "calculated_yield",
         "yield_difference"]
    ].head(10)
)


Valid yield checks: 6552

Mean absolute yield difference: 1.2718281644600185
Maximum absolute yield difference: 1526.125

Sample validation:


,crop,area_ha,production_tonnes,yield_kg_ha,calculated_yield,yield_difference
0,Sugarcane,0.07,1.10,15710.0,15714.285714,4.285714
1,Sugarcane,0.03,0.70,23329.0,23333.333333,4.333333
2,Arhar/Tur,0.80,0.65,810.0,812.500000,2.500000
4,Sugarcane,39.10,274.87,7030.0,7029.923274,0.076726
5,Sugarcane,40.75,321.84,7898.0,7897.914110,0.085890
6,Sugarcane,92.75,1979.86,21346.0,21346.199461,0.199461
7,Sugarcane,25.06,92.78,3702.0,3702.314445,0.314445
8,Jowar,313.00,1332.92,4259.0,4258.530351,0.469649
9,Jowar,167.88,639.49,3809.0,3809.208959,0.208959
10,Bajra,42.99,87.24,2029.0,2029.309142,0.309142


In [10]:
#Checking geography mapping
geo = pd.read_csv(upag_harmonized_file)

geo_map = (
    geo[
        ["state", "district", "lgd_statecode", "lgd_distcode"]
    ]
    .drop_duplicates()
)

print("Geography mapping rows:", len(geo_map))

print("\nDuplicate state-district mappings:")
print(
    geo_map.groupby(["state", "district"]).size()
    .loc[lambda x: x > 1]
)

Geography mapping rows: 782

Duplicate state-district mappings:
Series([], dtype: int64)


In [11]:
#Normalizing geougraphy names
def normalize_geo(x):
    return (
        str(x)
        .upper()
        .strip()
        .replace("&", "AND")
        .replace(".", "")
        .replace(",", "")
        .replace("-", " ")
    )

recent_long["state_key"] = recent_long["state"].apply(normalize_geo)
recent_long["district_key"] = recent_long["district"].apply(normalize_geo)

geo_map["state_key"] = geo_map["state"].apply(normalize_geo)
geo_map["district_key"] = geo_map["district"].apply(normalize_geo)

recent_long = recent_long.merge(
    geo_map[
        ["state_key", "district_key",
         "lgd_statecode", "lgd_distcode"]
    ],
    on=["state_key", "district_key"],
    how="left"
)

print("Total records:", len(recent_long))

print("Missing state codes:",
      recent_long["lgd_statecode"].isna().sum())

print("Missing district codes:",
      recent_long["lgd_distcode"].isna().sum())

Total records: 7372
Missing state codes: 96
Missing district codes: 96


In [12]:
#Showing unmatched districts
unmatched = recent_long[
    recent_long["lgd_distcode"].isna()
][
    ["state", "district"]
].drop_duplicates()

print("Unmatched geographic combinations:", len(unmatched))

if len(unmatched) > 0:
    display(unmatched.head(50))
else:
    print("All districts successfully matched.")

Unmatched geographic combinations: 10


,state,district
312,Andhra Pradesh,Y.S.R. Kadapa
886,Bihar,Kaimur (Bhabua)
1352,Chhattisgarh,Manendragarh-Chirmiri-Bharatpur(M C B)
3302,Madhya Pradesh,Khandwa (East Nimar)
3316,Madhya Pradesh,Khargone (West Nimar)
4530,Odisha,Angul
4542,Odisha,Balasore
4602,Odisha,Jajpur
4632,Odisha,Keonjhar
4786,Punjab,S.A.S Nagar


In [13]:
#Renaming codes and finalizing schema
recent_8_final = recent_long.rename(
    columns={
        "lgd_statecode": "state_code",
        "lgd_distcode": "district_code"
    }
).copy()

recent_8_final = recent_8_final[
    [
        "year",
        "state",
        "district",
        "state_code",
        "district_code",
        "crop",
        "season",
        "area_ha",
        "production_tonnes",
        "yield_kg_ha",
        "source"
    ]
]

print("Final schema:")
print(recent_8_final.columns.tolist())

print("\nShape:", recent_8_final.shape)

Final schema:
['year', 'state', 'district', 'state_code', 'district_code', 'crop', 'season', 'area_ha', 'production_tonnes', 'yield_kg_ha', 'source']

Shape: (7372, 11)


In [14]:
#Validating duplicates
key_columns = [
    "year",
    "state",
    "district",
    "crop"
]

duplicates = recent_8_final.duplicated(
    subset=key_columns
).sum()

print("Duplicate district-crop-year records:", duplicates)

if duplicates == 0:
    print("PASS: No duplicate records.")
else:
    print("WARNING: Duplicate records found.")
    display(
        recent_8_final[
            recent_8_final.duplicated(
                subset=key_columns,
                keep=False
            )
        ].sort_values(key_columns)
    )

Duplicate district-crop-year records: 0
PASS: No duplicate records.


In [15]:
#Coveraging validation for the 8 crops
coverage = (
    recent_8_final
    .groupby(["crop", "year"])
    .size()
    .unstack(fill_value=0)
)

print(coverage)

print("\nExpected crops:", len(target_crops))
print("Actual standardized crops:",
      recent_8_final["crop"].nunique())

print("\nExpected years: 2")
print("Actual years:",
      recent_8_final["year"].nunique())

year       2023-24  2024-25
crop                       
Arhar/Tur      550      550
Bajra          430      430
Gram           535      535
Groundnut      510      510
Jowar          432      432
Ragi           326      326
Soyabean       352      352
Sugarcane      551      551

Expected crops: 8
Actual standardized crops: 8

Expected years: 2
Actual years: 2


In [16]:
#Missing values check
print("Missing values:")
print(recent_8_final.isna().sum())

print("\nMissing percentage:")
print(
    (recent_8_final.isna().mean() * 100)
    .round(2)
)

Missing values:
year                   0
state                  0
district               0
state_code            96
district_code         96
crop                   0
season                 0
area_ha              793
production_tonnes    819
yield_kg_ha          819
source                 0
dtype: int64

Missing percentage:
year                  0.00
state                 0.00
district              0.00
state_code            1.30
district_code         1.30
crop                  0.00
season                0.00
area_ha              10.76
production_tonnes    11.11
yield_kg_ha          11.11
source                0.00
dtype: float64


In [17]:
#Saving the 8 crop upag extension
recent_8_final = recent_8_final.sort_values(
    ["year", "state", "district", "crop"]
).reset_index(drop=True)

recent_8_file = (
    output_dir / "upag_recent_8_crops_2023_2025.csv"
)

recent_8_final.to_csv(
    recent_8_file,
    index=False
)

print("Saved successfully.")
print("File:", recent_8_file)
print("Shape:", recent_8_final.shape)

Saved successfully.
File: D:\Projects\AgriRisk and ROI Prediction\Dump\data\processed\upag\upag_recent_8_crops_2023_2025.csv
Shape: (7372, 11)


In [18]:
#Loading existing unified data
unified = pd.read_csv(unified_file)

print("Existing unified dataset:")
print("Shape:", unified.shape)

print("\nSource counts:")
print(unified["source"].value_counts())

print("\nYears:")
print(sorted(unified["year"].unique()))

print("\nCrops:")
print(sorted(unified["crop"].unique()))

Existing unified dataset:
Shape: (60454, 11)

Source counts:
source
DES     54198
UPAg     6256
Name: count, dtype: int64

Years:
['2013-2014', '2014-2015', '2015-2016', '2016-2017', '2017-2018', '2018-2019', '2019-2020', '2020-2021', '2021-2022', '2022-2023', '2023-2024', '2024-2025']

Crops:
['Arhar/Tur', 'Bajra', 'Gram', 'Groundnut', 'Jowar', 'Maize', 'Ragi', 'Rice', 'Soyabean', 'Sugarcane', 'Urad', 'Wheat']


In [19]:
#Making Sure We Are Only Adding the Missing Recent Crops
existing_recent = unified[
    unified["year"].isin(["2023-24", "2024-25"])
]

print("Existing recent records:",
      len(existing_recent))

print("\nExisting recent crops:")
print(sorted(existing_recent["crop"].unique()))

print("\nNew 8-crop records:",
      len(recent_8_final))

print("\nNew crops:")
print(sorted(recent_8_final["crop"].unique()))

Existing recent records: 0

Existing recent crops:
[]

New 8-crop records: 7372

New crops:
['Arhar/Tur', 'Bajra', 'Gram', 'Groundnut', 'Jowar', 'Ragi', 'Soyabean', 'Sugarcane']


In [20]:
#Merging the 8 crops
unified_updated = pd.concat(
    [unified, recent_8_final],
    ignore_index=True
)

print("Old shape:", unified.shape)
print("New shape:", unified_updated.shape)

print("\nAdded records:",
      len(unified_updated) - len(unified))

Old shape: (60454, 11)
New shape: (67826, 11)

Added records: 7372


In [21]:
#Final duplicate check
final_keys = [
    "year",
    "state",
    "district",
    "crop"
]

final_duplicates = unified_updated.duplicated(
    subset=final_keys
).sum()

print("Final duplicate records:", final_duplicates)

if final_duplicates == 0:
    print("PASS: Final dataset has no duplicate district-crop-year records.")
else:
    print("WARNING: Duplicate records found.")
    display(
        unified_updated[
            unified_updated.duplicated(
                subset=final_keys,
                keep=False
            )
        ].sort_values(final_keys).head(50)
    )
    

Final duplicate records: 0
PASS: Final dataset has no duplicate district-crop-year records.


In [22]:
#Final crop year coverage
final_coverage = (
    unified_updated
    .groupby(["crop", "year"])
    .size()
    .unstack(fill_value=0)
)

print(final_coverage)

year       2013-2014  2014-2015  2015-2016  2016-2017  2017-2018  2018-2019  \
crop                                                                          
Arhar/Tur        484        455        457        523        527        537   
Bajra            293        290        274        283        307        340   
Gram             451        453        460        488        496        499   
Groundnut        402        380        396        422        457        480   
Jowar            315        299        296        310        356        361   
Maize            560        587        590        609        615        619   
Ragi             187        189        181        192        181        191   
Rice             591        607        612        631        642        649   
Soyabean         223        231        242        278        283        308   
Sugarcane        504        504        492        490        493        528   
Urad             492        505        517        56

In [23]:
#Strict 12 crops x 12 years check
expected_crops = [
    "Arhar/Tur",
    "Bajra",
    "Gram",
    "Groundnut",
    "Jowar",
    "Maize",
    "Ragi",
    "Rice",
    "Soyabean",
    "Sugarcane",
    "Urad",
    "Wheat"
]

expected_years = [
    "2013-2014",
    "2014-2015",
    "2015-2016",
    "2016-2017",
    "2017-2018",
    "2018-2019",
    "2019-2020",
    "2020-2021",
    "2021-2022",
    "2022-2023",
    "2023-2024",
    "2024-2025"
]

print("Expected crops:", len(expected_crops))
print("Actual crops:", unified_updated["crop"].nunique())

print("\nExpected years:", len(expected_years))
print("Actual years:", unified_updated["year"].nunique())

missing_combinations = []

for crop in expected_crops:
    for year in expected_years:
        count = len(
            unified_updated[
                (unified_updated["crop"] == crop) &
                (unified_updated["year"] == year)
            ]
        )

        if count == 0:
            missing_combinations.append(
                (crop, year)
            )

print("\nMissing crop-year combinations:",
      len(missing_combinations))

if missing_combinations:
    print(missing_combinations)
else:
    print("PASS: Every crop has data for every year.")

Expected crops: 12
Actual crops: 12

Expected years: 12
Actual years: 14

Missing crop-year combinations: 16
[('Arhar/Tur', '2023-2024'), ('Arhar/Tur', '2024-2025'), ('Bajra', '2023-2024'), ('Bajra', '2024-2025'), ('Gram', '2023-2024'), ('Gram', '2024-2025'), ('Groundnut', '2023-2024'), ('Groundnut', '2024-2025'), ('Jowar', '2023-2024'), ('Jowar', '2024-2025'), ('Ragi', '2023-2024'), ('Ragi', '2024-2025'), ('Soyabean', '2023-2024'), ('Soyabean', '2024-2025'), ('Sugarcane', '2023-2024'), ('Sugarcane', '2024-2025')]


In [24]:
#Source and year summary
print("Records by source:")
print(
    unified_updated["source"]
    .value_counts()
)

print("\nRecords by year and source:")
print(
    unified_updated
    .groupby(["year", "source"])
    .size()
    .unstack(fill_value=0)
)

Records by source:
source
DES     54198
UPAg    13628
Name: count, dtype: int64

Records by year and source:
source      DES  UPAg
year                 
2013-2014  5010     0
2014-2015  5001     0
2015-2016  5029     0
2016-2017  5307     0
2017-2018  5449     0
2018-2019  5618     0
2019-2020  5690     0
2020-2021  5682     0
2021-2022  5634     0
2022-2023  5778     0
2023-2024     0  3128
2023-24       0  3686
2024-2025     0  3128
2024-25       0  3686


In [25]:
#Final missing values
print("Final missing values:")
print(
    unified_updated[
        [
            "area_ha",
            "production_tonnes",
            "yield_kg_ha"
        ]
    ].isna().sum()
)

Final missing values:
area_ha              2322
production_tonnes    2480
yield_kg_ha          2355
dtype: int64


In [29]:
#Correcting year format + geography mapping

# Correcting remaining UPAg → LGD district names
district_mapping = {
    "YSR Kadapa": "Y S R Kadapa",
    "Kaimur": "Kaimur Bhabua",
    "Khandwa": "Khandwa East Nimar",
    "Khargone": "Khargone West Nimar",
    "Sahibzada Ajit Singh Nagar": "S A S Nagar"
}

# Recreating geography mapping
geo = pd.read_csv(upag_harmonized_file)

geo_map = (
    geo[
        ["state", "district", "lgd_statecode", "lgd_distcode"]
    ]
    .drop_duplicates()
)

geo_map["state_key"] = geo_map["state"].apply(normalize_geo)
geo_map["district_key"] = geo_map["district"].apply(normalize_geo)

# Applying corrected district names
recent_8_final["district"] = (
    recent_8_final["district"]
    .replace(district_mapping)
)

# Recreating matching keys
recent_8_final["state_key"] = (
    recent_8_final["state"].apply(normalize_geo)
)

recent_8_final["district_key"] = (
    recent_8_final["district"].apply(normalize_geo)
)

# Removing previous code columns
recent_8_final = recent_8_final.drop(
    columns=["state_code", "district_code"],
    errors="ignore"
)

# Merging LGD codes
recent_8_final = recent_8_final.merge(
    geo_map[
        [
            "state_key",
            "district_key",
            "lgd_statecode",
            "lgd_distcode"
        ]
    ],
    on=["state_key", "district_key"],
    how="left"
)

# Renaming codes
recent_8_final = recent_8_final.rename(
    columns={
        "lgd_statecode": "state_code",
        "lgd_distcode": "district_code"
    }
)

# Removing temporary columns
recent_8_final = recent_8_final[
    [
        "year",
        "state",
        "district",
        "state_code",
        "district_code",
        "crop",
        "season",
        "area_ha",
        "production_tonnes",
        "yield_kg_ha",
        "source"
    ]
]

print("Year values:")
print(sorted(recent_8_final["year"].unique()))

print("\nMissing state codes:",
      recent_8_final["state_code"].isna().sum())

print("Missing district codes:",
      recent_8_final["district_code"].isna().sum())

Year values:
['2023-2024', '2024-2025']

Missing state codes: 0
Missing district codes: 0


In [ ]:
#Building updated unified dataset

import pandas as pd

# Reloading original unified dataset
unified = pd.read_csv(
    r"D:\Projects\AgriRisk and ROI Prediction\Dump\data\processed\unified\unified_crop_yield_2013_2025.csv"
)

# Adding the corrected 8-crop UPAg data
unified_updated = pd.concat(
    [unified, recent_8_final],
    ignore_index=True
)

print("Original unified shape:", unified.shape)
print("Added recent 8-crop rows:", len(recent_8_final))
print("Updated unified shape:", unified_updated.shape)

print("\nYear values:")
print(sorted(unified_updated["year"].unique()))

Original unified shape: (60454, 11)
Added recent 8-crop rows: 7372
Updated unified shape: (67826, 11)

Year values:
['2013-2014', '2014-2015', '2015-2016', '2016-2017', '2017-2018', '2018-2019', '2019-2020', '2020-2021', '2021-2022', '2022-2023', '2023-2024', '2024-2025']


In [ ]:
# Checking for duplicate records

key_cols = [
    "year",
    "state",
    "district",
    "crop",
    "season"
]

duplicates = unified_updated[
    unified_updated.duplicated(subset=key_cols, keep=False)
]

print("Duplicate records:", len(duplicates))

if len(duplicates) > 0:
    print("\nSample duplicates:")
    display(duplicates.head(20))
else:
    print("No duplicate records found.")

Duplicate records: 0
No duplicate records found.


In [32]:
#Crop-Year Coverage Check(12 crops x 12 years)

coverage = (
    unified_updated
    .groupby(["crop", "year"])
    .size()
    .unstack(fill_value=0)
)

print("Crop-Year Coverage:")
display(coverage)

print("\nMissing crop-year combinations:")
missing = []

for crop in coverage.index:
    for year in coverage.columns:
        if coverage.loc[crop, year] == 0:
            missing.append((crop, year))

print("Total missing combinations:", len(missing))

if missing:
    for crop, year in missing:
        print(f"{crop} → {year}")
else:
    print("All crop-year combinations are present.")

Crop-Year Coverage:


year,2013-2014,2014-2015,2015-2016,2016-2017,2017-2018,2018-2019,2019-2020,2020-2021,2021-2022,2022-2023,2023-2024,2024-2025
crop,,,,,,,,,,,,
Arhar/Tur,484,455,457,523,527,537,547,538,526,519,550,550
Bajra,293,290,274,283,307,340,370,347,342,370,430,430
Gram,451,453,460,488,496,499,521,514,507,522,535,535
Groundnut,402,380,396,422,457,480,475,478,464,468,510,510
Jowar,315,299,296,310,356,361,370,369,370,377,432,432
Maize,560,587,590,609,615,619,641,633,644,655,782,782
Ragi,187,189,181,192,181,191,211,215,224,231,326,326
Rice,591,607,612,631,642,649,656,669,647,675,782,782
Soyabean,223,231,242,278,283,308,289,296,302,320,352,352



Missing crop-year combinations:
Total missing combinations: 0
All crop-year combinations are present.


In [ ]:
#Strict 12 × 12 Coverage Validation

expected_crops = [
    "Arhar/Tur",
    "Bajra",
    "Gram",
    "Groundnut",
    "Jowar",
    "Maize",
    "Ragi",
    "Rice",
    "Soyabean",
    "Sugarcane",
    "Urad",
    "Wheat"
]

expected_years = [
    "2013-2014",
    "2014-2015",
    "2015-2016",
    "2016-2017",
    "2017-2018",
    "2018-2019",
    "2019-2020",
    "2020-2021",
    "2021-2022",
    "2022-2023",
    "2023-2024",
    "2024-2025"
]

actual_crops = sorted(unified_updated["crop"].unique())
actual_years = sorted(unified_updated["year"].unique())

print("Number of crops:", len(actual_crops))
print("Number of years:", len(actual_years))
print("Expected crop-year combinations:", 12 * 12)
print("Actual crop-year combinations:",
      unified_updated.groupby(["crop", "year"]).ngroups)

print("\nMissing crops:", set(expected_crops) - set(actual_crops))
print("Extra crops:", set(actual_crops) - set(expected_crops))

print("\nMissing years:", set(expected_years) - set(actual_years))
print("Extra years:", set(actual_years) - set(expected_years))

if (
    len(actual_crops) == 12
    and len(actual_years) == 12
    and set(actual_crops) == set(expected_crops)
    and set(actual_years) == set(expected_years)
    and unified_updated.groupby(["crop", "year"]).ngroups == 144
):
    print("\n✓ STRICT 12 CROPS × 12 YEARS CHECK PASSED")
else:
    print("\n✗ STRICT COVERAGE CHECK FAILED")

Number of crops: 12
Number of years: 12
Expected crop-year combinations: 144
Actual crop-year combinations: 144

Missing crops: set()
Extra crops: set()

Missing years: set()
Extra years: set()

✓ STRICT 12 CROPS × 12 YEARS CHECK PASSED


In [34]:
#Final Dataset Summary

print("Dataset shape:", unified_updated.shape)

print("\nSource distribution:")
print(unified_updated["source"].value_counts())

print("\nCrop distribution:")
print(unified_updated["crop"].value_counts().sort_index())

print("\nData types:")
print(unified_updated.dtypes)

print("\nMissing values:")
print(unified_updated.isna().sum())

Dataset shape: (67826, 11)

Source distribution:
source
DES     54198
UPAg    13628
Name: count, dtype: int64

Crop distribution:
crop
Arhar/Tur    6213
Bajra        4076
Gram         5981
Groundnut    5442
Jowar        4287
Maize        7717
Ragi         2654
Rice         7943
Soyabean     3476
Sugarcane    6155
Urad         7071
Wheat        6811
Name: count, dtype: int64

Data types:
year                  object
state                 object
district              object
state_code             int64
district_code          int64
crop                  object
season                object
area_ha              float64
production_tonnes    float64
yield_kg_ha          float64
source                object
dtype: object

Missing values:
year                    0
state                   0
district                0
state_code              0
district_code           0
crop                    0
season                  0
area_ha              2322
production_tonnes    2480
yield_kg_ha          2355


In [35]:
#Saving Final Unified Dataset

output_path = (
    r"D:\Projects\AgriRisk and ROI Prediction\Dump"
    r"\data\processed\unified\unified_crop_yield_2013_2025.csv"
)

unified_updated.to_csv(output_path, index=False)

print("Final dataset saved successfully!")
print("Path:", output_path)
print("Shape:", unified_updated.shape)

Final dataset saved successfully!
Path: D:\Projects\AgriRisk and ROI Prediction\Dump\data\processed\unified\unified_crop_yield_2013_2025.csv
Shape: (67826, 11)
